# 🌦️ Weather Prediction — Step by Step

This notebook walks you through the **entire pipeline** from raw data to predictions.
Run each cell one by one and read the comments to understand what's happening.

## Step 1 — Install & Import Libraries

In [ ]:
# Run this cell first to install required packages
# (Only needed once per machine)
# !pip install -r ../requirements.txt

import sys
sys.path.insert(0, '..')   # So Python can find our src/ folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Our custom modules
from src.data_loader import load_rainfall, load_temperature, generate_sample_data
from src.features import build_rainfall_features, build_temperature_features

print('✅ All imports successful!')

## Step 2 — Load Data

**Before running this:**
- Download rainfall CSV from: https://www.data.gov.in/catalog/rainfall-india
- Download temperature data from: https://dsp.imdpune.gov.in/home_extremes.php
- Place both files in the `data/raw/` folder

If you don't have files yet, sample data will be generated automatically.

In [ ]:
import os
os.makedirs('../data/raw', exist_ok=True)

# Generate sample data if real files aren't available yet
if not os.path.exists('../data/raw/imd_rainfall.csv'):
    print('Real data not found — using sample data for now.')
    print('Replace data/raw files with your downloaded CSVs when ready.\n')
    os.chdir('..')
    generate_sample_data()
    os.chdir('notebooks')

os.chdir('..')
rain_df = load_rainfall()
temp_df = load_temperature()
os.chdir('notebooks')

print('\n--- Rainfall Data ---')
display(rain_df.head())

print('\n--- Temperature Data ---')
display(temp_df.head())

## Step 3 — Explore the Data (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Average rainfall by month across all stations
monthly_rain = rain_df.groupby('MONTH')['RAINFALL_MM'].mean()
axes[0].bar(monthly_rain.index, monthly_rain.values, color='steelblue')
axes[0].set_title('Average Monthly Rainfall (all stations)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Rainfall (mm)')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

# Average temperature by month
monthly_temp = temp_df.groupby('MONTH')['TEMPERATURE_C'].mean()
axes[1].plot(monthly_temp.index, monthly_temp.values, marker='o', color='orangered')
axes[1].fill_between(monthly_temp.index, monthly_temp.values, alpha=0.2, color='orangered')
axes[1].set_title('Average Monthly Temperature (all stations)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Rainfall trend over years
if 'YEAR' in rain_df.columns:
    yearly = rain_df.groupby('YEAR')['RAINFALL_MM'].sum()
    plt.figure(figsize=(12, 4))
    plt.plot(yearly.index, yearly.values, color='steelblue', linewidth=2)
    plt.title('Annual Total Rainfall Over Years')
    plt.xlabel('Year')
    plt.ylabel('Total Rainfall (mm)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 4 — Feature Engineering

In [ ]:
os.chdir('..')
rain_feat, rain_cols = build_rainfall_features(rain_df.copy())
temp_feat, temp_cols = build_temperature_features(temp_df.copy())
os.chdir('notebooks')

print('Rainfall feature columns:', rain_cols)
print('\nSample row:')
display(rain_feat[rain_cols].head(3))

## Step 5 — Train Models

In [ ]:
os.chdir('..')
from src.train import train_rainfall, train_temperature
rain_scores = train_rainfall()
temp_scores = train_temperature()
os.chdir('notebooks')

## Step 6 — Compare Model Scores

In [ ]:
print('🌧️ Rainfall Model Scores:')
display(rain_scores.sort_values('R2', ascending=False))

print('\n🌡️ Temperature Model Scores:')
display(temp_scores.sort_values('R2', ascending=False))

## Step 7 — Make Predictions

In [ ]:
from src.predict import predict_rainfall, predict_temperature, predict_year
os.chdir('..')

# Single prediction
station = 'INDORE'
month   = 7      # July
year    = 2025

rain = predict_rainfall(month, year, station)
temp = predict_temperature(month, year, station)

print(f'Predicted for {station}, July {year}:')
print(f'  Rainfall    : {rain} mm')
print(f'  Temperature : {temp} °C')

os.chdir('notebooks')

In [ ]:
os.chdir('..')

# Predict full year
yearly = predict_year(2025, 'INDORE')
display(yearly)

# Visualise
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(yearly['Month'], yearly['Rainfall (mm)'], color='steelblue', alpha=0.7, label='Rainfall')
ax2.plot(yearly['Month'], yearly['Temperature (°C)'], color='orangered', marker='o', linewidth=2, label='Temperature')

ax1.set_xlabel('Month')
ax1.set_ylabel('Rainfall (mm)', color='steelblue')
ax2.set_ylabel('Temperature (°C)', color='orangered')
plt.title('INDORE — 2025 Predicted Weather')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('outputs/yearly_forecast.png', dpi=120)
plt.show()
os.chdir('notebooks')